In [168]:
import torch
import torch.nn as nn

torch.manual_seed(42)

DATA PREPARATION!

In [169]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# dataset
df = pd.read_csv("fmnist_small.csv")

# # Plot the data and imshow
# fig, axes = plt.subplots(3, 4)
# for i in range(10):
#     img = df.iloc[i, 1:].values.reshape(28, 28)
#     axes.flat[i].imshow(img)

# split train and test
X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

# Scale only the pixels
X_train_scaled = X_train/255.0
X_test_scaled = X_test/255.0

# Convert all to tensors
'''
Note: 
1. X,y are Pandas DataFrame not NumpyArrays
2. Dataframe --> .values --> NumpyArray --> torch.tensor --> TorchTensors
'''
X_train_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.long)
'''
X_train_tensor = torch.from_numpy(X_train_scaled.values).float()
X_test_tensor = torch.from_numpy(X_test_scaled.values).float()
y_train_tensor = torch.from_numpy(y_train.values).long()
y_test_tensor = torch.from_numpy(y_test.values).long()
'''

'\nX_train_tensor = torch.from_numpy(X_train_scaled.values).float()\nX_test_tensor = torch.from_numpy(X_test_scaled.values).float()\ny_train_tensor = torch.from_numpy(y_train.values).long()\ny_test_tensor = torch.from_numpy(y_test.values).long()\n'

DATA LOADING!

In [170]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, X_train_tensor, y_train_tensor):
        self.X_train_tensor = X_train_tensor
        self.y_train_tensor = y_train_tensor
    def __len__(self):
        return len(self.X_train_tensor)
    def __getitem__(self,idx):
        return self.X_train_tensor[idx], self.y_train_tensor[idx]

train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

MODEL ARCHITECTURE!

In [171]:
class MyNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 10)
    )
  def forward(self, x):
    return self.model(x)

In [172]:
from torch import optim

# set learning rate and epochs
epochs = 100
learning_rate = 0.1

# instatiate the model
model = MyNN(X_train.shape[1])
# loss function
loss_fun = nn.CrossEntropyLoss()
# optimizer
optimizer = optim.SGD(model.parameters(), lr= learning_rate)

TRAINING LOOP!

In [173]:
# training loop
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_features, batch_labels in train_loader:
    # forward pass
    outputs = model(batch_features)
    # calculate loss
    loss = loss_fun(outputs, batch_labels)
    # back pass
    optimizer.zero_grad()
    loss.backward()
    # update grads
    optimizer.step()
    total_epoch_loss = total_epoch_loss + loss.item()
  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')


Epoch: 1 , Loss: 1.3354568270842235
Epoch: 2 , Loss: 0.7806735960642497
Epoch: 3 , Loss: 0.6551915393273036
Epoch: 4 , Loss: 0.5828122087319692
Epoch: 5 , Loss: 0.5376128798723221
Epoch: 6 , Loss: 0.4966834657390912
Epoch: 7 , Loss: 0.4559667379657428
Epoch: 8 , Loss: 0.44208129435777666
Epoch: 9 , Loss: 0.4213018386562665
Epoch: 10 , Loss: 0.39730726609627404
Epoch: 11 , Loss: 0.3861707626779874
Epoch: 12 , Loss: 0.3680653391778469
Epoch: 13 , Loss: 0.3651863776644071
Epoch: 14 , Loss: 0.3449504739046097
Epoch: 15 , Loss: 0.32258485138416293
Epoch: 16 , Loss: 0.31172908837596575
Epoch: 17 , Loss: 0.3116154084106286
Epoch: 18 , Loss: 0.2932840685794751
Epoch: 19 , Loss: 0.27472381472587587
Epoch: 20 , Loss: 0.26291155921916165
Epoch: 21 , Loss: 0.2668935678899288
Epoch: 22 , Loss: 0.25854284236828484
Epoch: 23 , Loss: 0.24679974528650442
Epoch: 24 , Loss: 0.2401453899592161
Epoch: 25 , Loss: 0.2335155006746451
Epoch: 26 , Loss: 0.21834628090262412
Epoch: 27 , Loss: 0.21294707983732222


In [ ]:
# set model to eval mode
model.eval()

# evaluation code
total = len(y_test_tensor)
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    outputs = model(batch_features)
    logit_values, predicted = torch.max(outputs, 1)
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.8558333333333333
